# 01b — Merge cleaned acquisitions and run FTH

Reload the separately cleaned output for every acquisition ID, average the selected IDs by polarization, then center the merged holograms, run FTH, and save one combined workflow data file.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

BASEFOLDER = Path.cwd().resolve()
sys.path.insert(0, str(BASEFOLDER / "library"))
import CCI_core as cci
import fthcore as fth
import fth_phase_workflow as wf
import helper_functions as helper
import interactive
import reconstruct_rb as rec
from mask_store import MaskStore

%matplotlib widget
print("Base folder:", BASEFOLDER)

## Configuration

In [ ]:
USER = "rb"
CLEANED_FOLDER = BASEFOLDER / "processed" / "cleaned_acquisitions"
IMAGE_IDS = {
    "+": [413],
    "-": [413],
}
MASK_ID = 95
CENTER_START = [1003, 1035]
ROI = [597, 785, 526, 719]
BUTTERWORTH_RADIUS = 35
BUTTERWORTH_ORDER = 4
PIXEL_MASK_SMOOTHING = 3

## Reload separate acquisitions, then merge them

In [ ]:
loaded_images = {"+": [], "-": []}
loaded_files = {"+": [], "-": []}
thresholds = {"+": [], "-": []}
dark_ids = None
energy = ccd_dist = px_size = None

for label, image_ids in IMAGE_IDS.items():
    for image_id in image_ids:
        input_file = CLEANED_FOLDER / f"cleaned_ImId_{image_id:04d}_{USER}.npz"
        with np.load(input_file, allow_pickle=False) as saved:
            saved_id = int(saved["image_id"])
            if saved_id != image_id:
                raise ValueError(f"Acquisition metadata does not match {input_file}")
            loaded_images[label].append(np.asarray(saved["image"], dtype=float))
            thresholds[label].append(float(saved["threshold"]))
            current_dark_ids = saved["dark_ids"].astype(int).tolist()
            current_setup = (
                float(saved["energy_eV"]),
                float(saved["ccd_dist_m"]),
                float(saved["px_size_m"]),
            )
        if dark_ids is None:
            dark_ids = current_dark_ids
            energy, ccd_dist, px_size = current_setup
        elif current_dark_ids != dark_ids or current_setup != (energy, ccd_dist, px_size):
            raise ValueError("Selected acquisitions do not share dark IDs and geometry")
        loaded_files[label].append(input_file)
        print(f"Loaded {label} ID {image_id}: {input_file}")

# This is the first point where separate acquisition IDs are merged.
positive = np.mean(np.stack(loaded_images["+"]), axis=0, dtype=np.float64)
negative = np.mean(np.stack(loaded_images["-"]), axis=0, dtype=np.float64)
positive_ids = [int(image_id) for image_id in IMAGE_IDS["+"]]
negative_ids = [int(image_id) for image_id in IMAGE_IDS["-"]]
if positive.shape != negative.shape or positive.ndim != 2:
    raise ValueError(f"Expected matching 2-D averages, got {positive.shape} and {negative.shape}")

im_id, topo_id = positive_ids[0], negative_ids[0]
folder_general = Path(helper.create_folder(BASEFOLDER / "processed"))
folder_logs = Path(helper.create_folder(folder_general / "Logs"))
DATA_H5 = folder_logs / f"data_recon_ImId_{im_id:04d}_{USER}.hdf5"
experimental_setup = {
    "ccd_dist": ccd_dist, "px_size": px_size, "binning": 1,
    "oversaturation": 60e3, "energy": energy,
    "lambda": helper.photon_energy_wavelength(energy, input_unit="eV"),
}
data = {
    "workflow": "FTH_from_cleaned_3d_averages",
    "user": USER, "data_file": str(DATA_H5),
    "experimental_setup": experimental_setup,
    "positive_label": "+", "reference_label": "-",
    "hologram_labels": ["+", "-"],
    "preprocessed_files": {
        label: [str(path) for path in paths] for label, paths in loaded_files.items()
    },
    "intensity_thresholds": thresholds,
    "holo": {
        "+": {"id": positive_ids, "dark_id": dark_ids, "image": positive},
        "-": {"id": negative_ids, "dark_id": dark_ids, "image": negative},
    },
}
print("Merged shapes:", positive.shape, negative.shape)
print("Output HDF5:", DATA_H5)

## Choose the detector center

In [ ]:
center_widget = interactive.InteractiveCenter(
    data["holo"]["-"]["image"], c0=CENTER_START[0], c1=CENTER_START[1]
)
# Adjust the widget before running the next cell.

In [ ]:
center = [center_widget.c0, center_widget.c1]
data["center"] = center
data = wf.define_centered_holograms(data, cci)
print("Center:", center)

## Load and center the detector mask

In [ ]:
mask_store = MaskStore(BASEFOLDER / "processed" / "mask_pixels")
mask_pixel_raw = mask_store.load(MASK_ID, positive.shape)
mask_pixel = (wf.center_image(mask_pixel_raw, center, cci) > 0.5).astype(np.uint8)
data["mask_pixel"] = mask_pixel
data["mask_pixel_raw_by_label"] = {"+": mask_pixel_raw, "-": mask_pixel_raw}
for state in data["holo"].values():
    state["mask_id_used"] = MASK_ID
    state["mask_pixel_raw"] = mask_pixel_raw
    state["mask_pixel_c"] = mask_pixel
print("Masked pixels:", int(mask_pixel.sum()))

## Construct and inspect the FTH hologram

In [ ]:
shape = positive.shape
mask_beamstop_smooth = wf.butterworth_disk_mask(shape, BUTTERWORTH_RADIUS, BUTTERWORTH_ORDER)
mask_pixel_fth = wf.smooth_binary_mask(
    mask_pixel.astype(float), PIXEL_MASK_SMOOTHING, PIXEL_MASK_SMOOTHING
)
mask_multiplier = (1 - mask_beamstop_smooth) * (1 - mask_pixel_fth)
pos = np.asarray(data["holo"]["+"]["image_c"], dtype=float)
neg = np.asarray(data["holo"]["-"]["image_c"], dtype=float)
factor, offset = cci.dyn_factor(
    pos * (1 - mask_pixel), neg * (1 - mask_pixel),
    method="correlation", verbose=False, plot=False,
)
holo_unmasked = pos / factor - neg - offset
holo_masked = holo_unmasked * mask_multiplier
data["factor"], data["offset"] = float(factor), float(offset)
data["mask_beamstop_smooth_recipe"] = {"radius": BUTTERWORTH_RADIUS, "order": BUTTERWORTH_ORDER}
data["mask_pixel_fth_recipe"] = {"dilation_pixels": PIXEL_MASK_SMOOTHING, "sigma": PIXEL_MASK_SMOOTHING}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, image, title in zip(axes, (pos, neg, holo_masked), ("+ average", "- average", "masked difference")):
    vmin, vmax = wf.finite_percentile_limits(image, (1, 99.9))
    axis.imshow(image, vmin=vmin, vmax=vmax, cmap="viridis")
    axis.set_title(title)
    axis.set_axis_off()
plt.tight_layout(); plt.show()

## Focus and crop the FTH reconstruction

In [ ]:
roi_s = np.s_[ROI[0]:ROI[1], ROI[2]:ROI[3]]
prop_dist, phase, dx, dy = 0.0, 0.0, 0.0, 0.0
focus_sliders = rec.focusCDI(
    holo_masked, np.zeros_like(holo_masked), roi_s, mask=1,
    phase=phase, prop_dist=prop_dist, dx=dx, dy=dy,
    experimental_setup=experimental_setup, operation="-",
    max_prop_dist=30, scale=(2, 98),
)
slider_prop, slider_phase, slider_dx, slider_dy = focus_sliders[:4]

## Apply the selected focus and save

In [ ]:
prop_dist = float(slider_prop.value)
phase = float(slider_phase.value)
dx = float(slider_dx.value)
dy = float(slider_dy.value)
reconstruction = wf.fth_reconstruct(
    holo_masked, experimental_setup, fth,
    prop_dist=prop_dist, phase=phase, dx=dx, dy=dy,
)
data["focus_fth"] = {
    "prop_dist": prop_dist, "prop_dist_unit": "um",
    "phase": phase, "dx": dx, "dy": dy,
    "roi": ROI, "operation": "-",
}
data["recon"] = reconstruction[roi_s]
png_name = folder_general / f"FTH_recon_ImId_{im_id:04d}_{USER}.png"
fig, axis = plt.subplots(figsize=(5, 5))
shown = np.real(data["recon"])
vmin, vmax = wf.finite_percentile_limits(shown)
axis.imshow(shown, vmin=vmin, vmax=vmax, cmap="gray")
axis.set_title(f"Focused FTH — averaged IDs beginning at {im_id}")
axis.set_axis_off()
fig.savefig(png_name, bbox_inches="tight", dpi=200)
plt.show()
data["fth_png"] = str(png_name)
wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Saved HDF5:", DATA_H5)
print("Saved PNG:", png_name)

In [ ]:
print("im_ids:", {"+": positive_ids, "-": negative_ids})
print("dark_ids:", dark_ids)